# BumbaLM — avaliação no teste dos modelos treinados no node2 (A100, TF32)

Este notebook fecha o experimento com o que já está treinado:
- avalia no teste completo (9.999 consultas, 92.101 trechos) os dois modelos do **ajuste completo** treinados no node2, **nenhum** e **difíceis**, e o **modelo base**;
- os três rodam na **mesma precisão (TF32)** e na mesma placa (A100), para a comparação ser justa;
- no fim, monta a tabela do TCC com o **BM25**, a divisão entre consultas com e sem gêmeos, os **testes de significância pareados** e as curvas de validação do treino.

A variante **aleatórios** não terminou no node2 e fica fora.

**Custo esperado:** ~2–3 h de A100 por modelo, ~7 h no total (~60 unidades a 8,88 por hora). A célula 6 mede a velocidade real antes de começar.

### Antes da primeira sessão (uma vez)

Ponha no Google Drive, dentro de `MyDrive/BumbaLM_colab/`:

| Pasta no Drive | O que | De onde vem |
|---|---|---|
| `dados/` | `processed_tjrs100k.tar.gz`, `modelo_revisao_node2.txt`, `pip_freeze_node2.txt` | já está lá (pacote de 07/11) |
| `modelos_node2/nenhum/` | a pasta `best_model_4b/` inteira e o `melhor_checkpoint.json`; se tiver, o `metricas_treino_val.csv` | Tapajós: `/projetos/ctcca/nlp/bumbalm/h200_tjrs_tcc/tcc_tjrs100k_ablacao_negativos_h200/nenhum/` |
| `modelos_node2/dificeis/` | o mesmo, da difíceis | Tapajós: `…/tcc_tjrs100k_ablacao_negativos_h200/dificeis/` |
| `modelos_node2/` | `SHA256SUMS_nenhum_dificeis.txt` | Tapajós: `…/tcc_tjrs100k_ablacao_negativos_h200/` |
| `resultados_externos/bm25/` | `metricas_test_bm25.json` e `por_consulta_test_bm25.jsonl` | seu PC: `Downloads\BumbaLM_teste100k\avaliacao_bm25\` |
| `resultados_externos/base_fp32_node2/` | `metricas_test_baseline.json` e `por_consulta_test_baseline.jsonl` | seu PC: `Downloads\diagnostico\h200_100k\experiments\diagnostico_h200_100k\base_teste\test\baseline\` |

Os dois modelos têm ~16 GB cada. Se não couberem no Drive, suba-os para um repositório **privado** do Hugging Face com a mesma estrutura (`nenhum/…`, `dificeis/…`, `SHA256SUMS_nenhum_dificeis.txt`) e use `ORIGEM_MODELOS = "hf"` na célula 4b.

*Ambiente de execução → Alterar o tipo de ambiente de execução → **A100 GPU**, com RAM alta.*

### Em toda sessão (inclusive depois de uma queda)

Rode as células **1 a 4b** e depois a célula de onde parou. O que já foi avaliado fica no Drive e é pulado.

## 1. GPU e Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

## 2. Código

Clona a branch do GitHub. Se o repositório for privado, crie o segredo `GITHUB_TOKEN` (ícone da chave, à esquerda) com um *token* de leitura.
Sem GitHub, suba um `codigo.zip` do repositório para `MyDrive/BumbaLM_colab/` e ponha `USAR_ZIP = True`.

In [ ]:
import os, subprocess, zipfile
REPO_GITHUB = "alekxDias/BumbaLM-Embedding-4B-Train"
BRANCH = "tcc/ablacao-negativos-tjrs-h200"
USAR_ZIP = False
DESTINO = "/content/BumbaLM"
DRIVE = "/content/drive/MyDrive/BumbaLM_colab"

if USAR_ZIP:
    if not os.path.isdir(DESTINO):
        with zipfile.ZipFile(f"{DRIVE}/codigo.zip") as z:
            z.extractall("/content/BumbaLM_zip")
        raiz = next(r for r, d, f in os.walk("/content/BumbaLM_zip") if "colab" in d and "src" in d)
        os.rename(raiz, DESTINO)
elif os.path.isdir(f"{DESTINO}/.git"):
    subprocess.run(["git", "-C", DESTINO, "pull", "--ff-only"], check=True)
else:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    url = f"https://{token}@github.com/{REPO_GITHUB}.git" if token else f"https://github.com/{REPO_GITHUB}.git"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, DESTINO], check=True)

os.chdir(DESTINO)
os.environ["BUMBALM_DRIVE"] = DRIVE
!git log --oneline -1 2>/dev/null || echo "(codigo do zip)"

## 3. Dependências

As mesmas versões do node2 (`pip_freeze_node2.txt`). O `torch` do Colab é mantido.

In [ ]:
import os, subprocess, sys
PACOTES = ["sentence-transformers", "transformers", "peft", "accelerate", "datasets", "huggingface-hub"]
congelado = f"{DRIVE}/dados/pip_freeze_node2.txt"
versoes = {}
if os.path.isfile(congelado):
    for linha in open(congelado, encoding="utf-8", errors="replace"):
        if "==" in linha:
            nome, ver = linha.strip().split("==", 1)
            versoes[nome.lower().replace("_", "-")] = ver
pedidos = [f"{p}=={versoes[p]}" if p in versoes else p for p in PACOTES]
if "sentence-transformers" not in versoes:
    pedidos = ["sentence-transformers>=5.0,<6", "transformers>=4.51", "peft>=0.15", "accelerate>=1.0", "datasets>=3.0", "huggingface-hub"]
print("Instalando:", " ".join(pedidos))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pedidos, "pyyaml", "psutil", "pyarrow"], check=True)
# O Colab traz um torchao antigo que atrapalha o peft; o pipeline nao usa torchao.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)

## 4. Dados de teste e modelo base

Pula o que já estiver pronto nesta sessão:
- descompacta os dados no disco local e confere o SHA-256 de cada arquivo com o do node2;
- baixa o Qwen3-Embedding-4B na **mesma revisão do node2**.

In [ ]:
!python colab/rodar.py preparar

### 4b. Caminhos, opções e funções

Rode em toda sessão. As opções em maiúsculas podem ser ajustadas aqui, **antes** do primeiro modelo; depois, não mude `PRECISAO` nem a placa.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/BumbaLM_colab")
REPO = Path("/content/BumbaLM")
DADOS = Path("/content/dados/processed_tjrs100k")       # descompactado na celula 4
BASE = Path("/content/modelos/Qwen3-Embedding-4B")       # baixado na celula 4, na revisao do node2
MODELOS_DRIVE = DRIVE / "modelos_node2"
MODELOS_LOCAL = Path("/content/modelos_node2")
EXTERNOS = DRIVE / "resultados_externos"
SAIDA = DRIVE / "avaliacao_node2_tf32"                   # metricas e arquivos por consulta (no Drive)
FINAL = DRIVE / "resultados_finais"                      # tabela, testes e figuras do TCC
LOGS = DRIVE / "logs"

PRECISAO = "tf32"         # a mesma nos tres modelos
BATCH = 128               # o mesmo do node2; com falta de memoria, 64 (nao muda o resultado)
GPU_ESPERADA = "A100"     # todas as sessoes na mesma placa
UNIDADES_POR_HORA = 8.88  # confira no painel Recursos do Colab
ORIGEM_MODELOS = "drive"  # "drive" ou "hf"
HF_REPO = None            # com "hf": ex. "seu-usuario/bumbalm-tcc-full" (privado) e o segredo HF_TOKEN
VARIANTES = ["nenhum", "dificeis"]
LISTA_SHA = "SHA256SUMS_nenhum_dificeis.txt"
os.chdir(REPO)


def nome_gpu():
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    return r.stdout.strip().splitlines()[0] if r.returncode == 0 and r.stdout.strip() else ""


def conferir_gpu():
    nome = nome_gpu()
    if not nome:
        raise SystemExit("Sem GPU nesta sessao. Ambiente de execucao -> Alterar o tipo -> A100 GPU.")
    if GPU_ESPERADA and GPU_ESPERADA not in nome:
        raise SystemExit(f"Esta sessao pegou '{nome}', nao uma {GPU_ESPERADA}. Troque o ambiente: "
                         "os tres modelos precisam ser avaliados na mesma placa e na mesma precisao.")
    print(f"GPU: {nome}")


def sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(16 * 1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()


def _token_hf():
    from google.colab import userdata
    return userdata.get("HF_TOKEN")


def ler_lista_sha():
    # {caminho relativo: sha256}, como no SHA256SUMS gerado no node2
    if ORIGEM_MODELOS == "hf":
        from huggingface_hub import hf_hub_download
        lista = Path(hf_hub_download(HF_REPO, LISTA_SHA, token=_token_hf(), local_dir=str(MODELOS_LOCAL)))
    else:
        lista = MODELOS_DRIVE / LISTA_SHA
    if not lista.is_file():
        raise SystemExit(f"Falta {lista}: suba o {LISTA_SHA} junto com os modelos.")
    esperado = {}
    for linha in lista.read_text(encoding="utf-8").splitlines():
        if linha.strip():
            h, nome = linha.split(maxsplit=1)
            esperado[nome.lstrip("*").strip()] = h
    return esperado


def arquivos_exigidos(variante, esperado):
    exigidos = {n: h for n, h in esperado.items()
                if n.startswith(f"{variante}/best_model_4b/") or n == f"{variante}/melhor_checkpoint.json"}
    if not any(n.startswith(f"{variante}/best_model_4b/") for n in exigidos):
        raise SystemExit(f"O {LISTA_SHA} nao lista arquivos de {variante}/best_model_4b.")
    return exigidos


def preparar_variante(variante):
    # Copia o modelo para o disco local e confere o SHA-256 de cada arquivo com o do node2
    destino = MODELOS_LOCAL / variante
    marcador = destino / ".conferido"
    if marcador.is_file():
        print(f"{variante}: ja copiado e conferido nesta sessao.")
        return
    esperado = ler_lista_sha()
    exigidos = arquivos_exigidos(variante, esperado)
    destino.mkdir(parents=True, exist_ok=True)
    inicio = time.time()
    if ORIGEM_MODELOS == "hf":
        from huggingface_hub import snapshot_download
        print(f"{variante}: baixando de {HF_REPO} (~16 GB)...", flush=True)
        snapshot_download(repo_id=HF_REPO, token=_token_hf(), local_dir=str(MODELOS_LOCAL),
                          allow_patterns=[f"{variante}/best_model_4b/*", f"{variante}/melhor_checkpoint.json"])
    else:
        origem = MODELOS_DRIVE / variante
        print(f"{variante}: copiando {origem / 'best_model_4b'} para o disco local (~16 GB, alguns minutos)...", flush=True)
        shutil.copytree(origem / "best_model_4b", destino / "best_model_4b", dirs_exist_ok=True)
        shutil.copy2(origem / "melhor_checkpoint.json", destino / "melhor_checkpoint.json")
    print(f"  pronto em {time.time() - inicio:.0f} s. Conferindo o SHA-256 de {len(exigidos)} arquivos...", flush=True)
    falhas = []
    for nome, h in sorted(exigidos.items()):
        alvo = MODELOS_LOCAL / nome
        if not alvo.is_file():
            falhas.append(f"{nome} (ausente)")
        elif sha256(alvo) != h:
            falhas.append(f"{nome} (SHA-256 diferente)")
    if falhas:
        raise SystemExit(f"{variante}: arquivos diferentes dos do node2:\n  " + "\n  ".join(falhas))
    print(f"  {len(exigidos)} arquivos iguais aos do node2 ({time.time() - inicio:.0f} s no total).")
    marcador.write_text(json.dumps({"arquivos": len(exigidos)}), encoding="utf-8")


def comando_avaliacao(*extra):
    # Configs do node2 (mesmos 768 tokens, instrucao e lote); dados, modelos e precisao passados aqui
    return [sys.executable, "-u", "scripts/05_evaluate.py",
            "--config_base", "configs/config_base.yaml", "--config_train", "configs/config_train.yaml",
            "--data_dir", str(DADOS), "--baseline_id", str(BASE), "--compare_baseline",
            *[a for v in VARIANTES for a in ("--modelo", f"{v}={MODELOS_LOCAL / v / 'best_model_4b'}")],
            "--precisao", PRECISAO, "--batch_size", str(BATCH), "--split", "test",
            "--output_dir", str(SAIDA), "--pular_avaliados", *extra]


def rodar(comando, nome_log):
    LOGS.mkdir(parents=True, exist_ok=True)
    caminho_log = LOGS / f"{nome_log}_{time.strftime('%Y%m%d_%H%M%S')}.log"
    env = os.environ.copy()
    env.update({"PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false", "HF_HUB_OFFLINE": "1",
                "TRANSFORMERS_OFFLINE": "1", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
                # Barra de progresso a cada minuto, para a saida da celula nao ficar enorme
                "TQDM_MININTERVAL": "60"})
    env["PYTHONPATH"] = str(REPO) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    print(f"$ {' '.join(comando)}\n(log: {caminho_log})", flush=True)
    inicio = time.time()
    with open(caminho_log, "w", encoding="utf-8") as log:
        log.write(f"GPU: {nome_gpu()}\n")
        processo = subprocess.Popen(comando, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                    text=True, encoding="utf-8", errors="replace", bufsize=1)
        for linha in processo.stdout:
            sys.stdout.write(linha)
            sys.stdout.flush()
            log.write(linha)
        codigo = processo.wait()
    horas = (time.time() - inicio) / 3600
    print(f"\nTempo: {horas:.2f} h (~{horas * UNIDADES_POR_HORA:.0f} unidades)")
    if codigo != 0:
        raise SystemExit(f"Terminou com erro (codigo {codigo}). Log: {caminho_log}. Rode a celula de novo para retomar.")


def arquivo_metricas(nome):
    return SAIDA / "test" / nome / f"metricas_test_{nome}.json"


def avaliar(nome):
    pronto = arquivo_metricas(nome)
    if pronto.is_file():
        print(f"'{nome}' ja foi avaliado: {pronto}")
        return
    conferir_gpu()
    if nome in VARIANTES:
        preparar_variante(nome)
    rodar(comando_avaliacao("--somente", nome, "--sem_comparativo"), f"avaliar_{nome}")


def andamento():
    for nome in ["baseline", *VARIANTES]:
        arq = arquivo_metricas(nome)
        if arq.is_file():
            dados = json.loads(arq.read_text(encoding="utf-8"))
            m = dados.get("metricas_global", {})
            cfg = dados.get("configuracao_avaliacao", {})
            print(f"  {nome:<10} avaliado | nDCG@10 {m.get('ndcg@10', float('nan')):.4f} | "
                  f"MRR@10 {m.get('mrr@10', float('nan')):.4f} | precisao {cfg.get('precisao', '-')}")
        else:
            print(f"  {nome:<10} falta avaliar")


print("Configuracao carregada. Andamento:")
andamento()

## 5. Conferir os modelos no Drive (rápido)

Confere, sem ler os 32 GB, que estão no Drive todos os arquivos que o node2 listou para os dois modelos, e mostra o registro do treino de cada um.
As duas variantes têm de ter o **mesmo hash das tuplas** (mesmos pares de treino). A cópia e a conferência byte a byte do SHA-256 acontecem antes de avaliar cada uma (células 8 e 9).

In [ ]:
esperado = ler_lista_sha()
hashes = {}
for v in VARIANTES:
    exigidos = arquivos_exigidos(v, esperado)
    if ORIGEM_MODELOS == "hf":
        from huggingface_hub import HfApi, hf_hub_download
        no_repo = set(HfApi().list_repo_files(HF_REPO, token=_token_hf()))
        faltando = [n for n in exigidos if n not in no_repo]
        registro_arq = Path(hf_hub_download(HF_REPO, f"{v}/melhor_checkpoint.json", token=_token_hf(), local_dir=str(MODELOS_LOCAL)))
    else:
        faltando = [n for n in exigidos if not (MODELOS_DRIVE / n).is_file()]
        registro_arq = MODELOS_DRIVE / v / "melhor_checkpoint.json"
    if faltando:
        raise SystemExit(f"{v}: faltam {len(faltando)} arquivos, ex.: {faltando[:3]}")
    if ORIGEM_MODELOS != "hf":
        tamanho = sum((MODELOS_DRIVE / n).stat().st_size for n in exigidos) / 1024**3
        print(f"{v}: {len(exigidos)} arquivos no Drive, {tamanho:.1f} GB")
    else:
        print(f"{v}: {len(exigidos)} arquivos no Hugging Face")
    r = json.loads(registro_arq.read_text(encoding="utf-8"))
    hashes[v] = r.get("hash_tuplas_sorteadas")
    print(f"  tuplas {r.get('n_tuplas_treino')} | lote {r.get('batch_size_por_gpu')} x {r.get('num_gpus')} GPU(s) | "
          f"LR {r.get('learning_rate')} | melhor {r.get('metrica_selecao')} = {r.get('melhor_metrica')} | "
          f"tensores conferidos {r.get('tensores_conferidos_melhor_checkpoint')}")
    print(f"  hash das tuplas: {hashes[v]}")
if len(set(hashes.values())) != 1:
    raise SystemExit("As variantes treinaram com pares diferentes: nao da para comparar.")
print("\nOK: as duas variantes treinaram com os mesmos pares.")

## 6. Teste de velocidade (~5 min, recomendado na primeira sessão)

Carrega o modelo base em TF32, vetoriza 1.024 trechos do teste e estima horas e unidades. Também confere que o lote de 128 cabe na memória.

In [ ]:
conferir_gpu()
rodar([sys.executable, "-u", "colab/medir_velocidade_avaliacao.py", "--modelo", str(BASE),
       "--chunks", str(DADOS / "test_corpus_chunks.jsonl"), "--precisao", PRECISAO,
       "--batch_size", str(BATCH), "--unidades_por_hora", str(UNIDADES_POR_HORA)], "velocidade")

## 7. Modelo base em TF32

Primeiro o base: ele serve de referência e testa o fluxo inteiro antes de gastar com as variantes.

In [ ]:
avaliar("baseline")

### 7b. Conferência: base em TF32 × base em fp32 do node2

É o mesmo modelo no mesmo teste; muda só a precisão (e a placa). As médias devem ficar muito próximas (diferença na 3ª ou 4ª casa).
Uma diferença acima de 0,005 no nDCG@10 indica algum problema (dados, modelo ou configuração): pare e investigue antes de avaliar as variantes.

In [ ]:
ref = EXTERNOS / "base_fp32_node2"
if not ref.is_dir():
    print(f"Sem {ref}: suba os arquivos do base em fp32 para fazer a conferencia.")
else:
    saida_conf = FINAL / "conferencia_base_tf32_fp32"
    rodar([sys.executable, "-u", "scripts/06_comparar_resultados.py",
           "--resultado", f"base_tf32={SAIDA / 'test' / 'baseline'}", "--resultado", f"base_fp32_node2={ref}",
           "--pares", "base_tf32:base_fp32_node2", "--saida", str(saida_conf)], "conferencia_base")
    testes = json.loads((saida_conf / "resultados.json").read_text(encoding="utf-8"))["testes_pareados"]
    dif = next(t["diferenca"] for t in testes if t["metrica"] == "ndcg@10")
    print("\nOK: base em TF32 igual ao do node2 na pratica." if abs(dif) < 0.005
          else f"\nALERTA: diferenca de {dif:+.4f} no nDCG@10. Investigue antes de seguir.")

## 8. Variante nenhum

Copia o modelo para o disco local, confere o SHA-256 com o do node2 e avalia. Se a sessão cair, rode as células 1–4b e esta de novo.

In [ ]:
avaliar("nenhum")

## 9. Variante difíceis

In [ ]:
avaliar("dificeis")

## 10. Comparativo da avaliação (sem GPU, ~2 min)

Junta os três modelos no `comparativo_test.txt` e no `avaliacao_consolidada.json`. Antes, confere que os três resultados são do mesmo teste e da mesma configuração.

In [ ]:
faltando = [n for n in ["baseline", *VARIANTES] if not arquivo_metricas(n).is_file()]
if faltando:
    raise SystemExit(f"Ainda falta avaliar: {', '.join(faltando)}")
rodar(comando_avaliacao(), "comparativo")

## 11. Tabela final, gêmeos e testes de significância (sem GPU, ~1 min)

Junta o base, o BM25, a nenhum e a difíceis:
- tabela com nDCG@10, MRR@10, MAP@100, acurácia@1/5/10 e recall@10;
- as mesmas médias em consultas **sem** e **com** gêmeos (quase duplicatas relevantes);
- testes pareados por consulta em nDCG@10 e MRR@10: diferença com IC de 95% (bootstrap), p-valor do teste de aleatorização e p ajustado por Holm;
- curvas de validação do treino, se os `metricas_treino_val.csv` estiverem no Drive.

Tudo fica em `MyDrive/BumbaLM_colab/resultados_finais/`.

In [ ]:
faltando = [n for n in ["baseline", *VARIANTES] if not arquivo_metricas(n).is_file()]
if faltando:
    raise SystemExit(f"Ainda falta avaliar: {', '.join(faltando)}")
comando = [sys.executable, "-u", "scripts/06_comparar_resultados.py",
           "--resultado", f"base={SAIDA / 'test' / 'baseline'}"]
if (EXTERNOS / "bm25").is_dir():
    comando += ["--resultado", f"bm25={EXTERNOS / 'bm25'}"]
else:
    print(f"Sem {EXTERNOS / 'bm25'}: o BM25 fica fora da tabela.")
comando += ["--resultado", f"nenhum={SAIDA / 'test' / 'nenhum'}", "--resultado", f"dificeis={SAIDA / 'test' / 'dificeis'}"]
pares = ["dificeis:nenhum", "dificeis:base", "nenhum:base"]
if (EXTERNOS / "bm25").is_dir():
    pares += ["dificeis:bm25", "nenhum:bm25", "base:bm25"]
comando += ["--pares", ",".join(pares), "--saida", str(FINAL)]
for v in VARIANTES:
    curva = MODELOS_DRIVE / v / "metricas_treino_val.csv"
    if curva.is_file():
        comando += ["--curva", f"{v}={curva}"]
rodar(comando, "resultados_finais")

from IPython.display import Image, Markdown, display
display(Markdown((FINAL / "resultados.md").read_text(encoding="utf-8")))
if (FINAL / "curvas_validacao.png").is_file():
    display(Image(filename=str(FINAL / "curvas_validacao.png")))

## 12. Andamento

Mostra o que já foi avaliado. Os resultados ficam no Drive:
- `avaliacao_node2_tf32/test/<modelo>/`: métricas, relatório e arquivo por consulta de cada modelo;
- `avaliacao_node2_tf32/comparativo_test.txt`: o comparativo da avaliação;
- `resultados_finais/`: `resultados.md`, `tabela_metricas.csv`, `tabela_gemeos.csv`, `testes_pareados.csv`, `curvas_validacao.png/.pdf`;
- `logs/`: um log por execução.

In [ ]:
andamento()